# Dataset Exploration

This notebook explores the two parking space detection datasets:
1. **Roboflow Parking Lot (new)** - YOLO format
2. **HuggingFace UniqueData/parking-space-detection-dataset** - XML annotations


In [ ]:
import os
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = BASE_DIR / "data" / "raw"

rf_root = DATA_RAW / "roboflow_parking_lot_new"
hf_root = DATA_RAW / "hf_parking_space"

print(f"BASE_DIR: {BASE_DIR}")
print(f"Roboflow root exists: {rf_root.exists()}")
print(f"HuggingFace root exists: {hf_root.exists()}")


## 5. EDA for Roboflow Dataset (YOLO Format)


### 5.1 Count images & labels per split


In [ ]:
def count_yolo_split(split_dir: Path):
    img_dir = split_dir / "images"
    label_dir = split_dir / "labels"
    images = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg"))
    labels = list(label_dir.glob("*.txt"))
    return len(images), len(labels)

for split in ["train", "valid", "test"]:
    split_dir = rf_root / split
    if split_dir.exists():
        n_img, n_lbl = count_yolo_split(split_dir)
        print(f"{split}: {n_img} images, {n_lbl} label files")


### 5.2 Load class names from data.yaml


In [ ]:
import yaml

data_yaml_path = rf_root / "data.yaml"
with open(data_yaml_path, "r") as f:
    rf_cfg = yaml.safe_load(f)

rf_classes = rf_cfg.get("names", [])
print("Roboflow classes:", rf_classes)


### 5.3 Class distribution (across all splits)


In [ ]:
from collections import Counter

rf_class_counts = Counter()

for split in ["train", "valid", "test"]:
    label_dir = rf_root / split / "labels"
    if not label_dir.exists():
        continue
    for txt_path in label_dir.glob("*.txt"):
        with open(txt_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                rf_class_counts[cls_id] += 1

print("Class counts (by id):", rf_class_counts)
print("Class counts (by name):")
for cls_id, count in rf_class_counts.items():
    name = rf_classes[cls_id] if cls_id < len(rf_classes) else f"id_{cls_id}"
    print(f"  {cls_id} ({name}): {count}")


### 5.4 Visual sanity check (plot a few images + boxes)


In [ ]:
def plot_yolo_sample(split="train", n_samples=3):
    img_dir = rf_root / split / "images"
    label_dir = rf_root / split / "labels"
    img_paths = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png"))
    img_paths = img_paths[:n_samples]

    for img_path in img_paths:
        label_path = label_dir / (img_path.stem + ".txt")
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape

        if label_path.exists():
            with open(label_path, "r") as f:
                for line in f:
                    cls_id, x_c, y_c, bw, bh = map(float, line.strip().split())
                    x_c, y_c, bw, bh = x_c * w, y_c * h, bw * w, bh * h
                    x1 = int(x_c - bw / 2)
                    y1 = int(y_c - bh / 2)
                    x2 = int(x_c + bw / 2)
                    y2 = int(y_c + bh / 2)
                    color = (0, 255, 0)
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                    label = rf_classes[int(cls_id)] if int(cls_id) < len(rf_classes) else str(int(cls_id))
                    cv2.putText(img, label, (x1, max(0, y1 - 5)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"{split} sample: {img_path.name}")
        plt.show()

plot_yolo_sample("train", n_samples=3)


### 6.1 Locate paths


In [ ]:
hf_data_dir = hf_root / "data"
hf_images_dir = hf_data_dir / "images"
hf_boxes_dir = hf_data_dir / "boxes"
hf_annot_path = hf_root / "annotations.xml"

print(f"Images dir exists: {hf_images_dir.exists()}")
print(f"Boxes dir exists: {hf_boxes_dir.exists()}")
print(f"Annotations file exists: {hf_annot_path.exists()}")


### 6.2 Parse annotations.xml into a DataFrame

**Note:** The HF dataset uses polygon annotations. We'll convert polygons to bounding boxes by taking min/max coordinates.


In [ ]:
def parse_hf_annotations(xml_path: Path):
    """
    Parse CVAT XML format with polygon annotations.
    Converts polygons to bounding boxes.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    records = []
    for img_elem in root.findall("image"):
        img_name = img_elem.get("name")  # e.g., "images/0.png"
        img_w = int(img_elem.get("width"))
        img_h = int(img_elem.get("height"))

        # Handle polygon annotations
        for polygon in img_elem.findall("polygon"):
            label = polygon.get("label")
            points_str = polygon.get("points")
            
            # Parse points: "x1,y1;x2,y2;x3,y3;x4,y4"
            points = []
            for point_pair in points_str.split(";"):
                if point_pair.strip():
                    x, y = map(float, point_pair.split(","))
                    points.append((x, y))
            
            if len(points) < 2:
                continue
            
            # Convert polygon to bounding box
            x_coords = [p[0] for p in points]
            y_coords = [p[1] for p in points]
            x_min = min(x_coords)
            y_min = min(y_coords)
            x_max = max(x_coords)
            y_max = max(y_coords)
            
            # Clip to image bounds
            x_min = max(0, int(x_min))
            y_min = max(0, int(y_min))
            x_max = min(img_w, int(x_max))
            y_max = min(img_h, int(y_max))

            records.append({
                "image": img_name,
                "width": img_w,
                "height": img_h,
                "label": label,
                "x_min": x_min,
                "y_min": y_min,
                "x_max": x_max,
                "y_max": y_max,
            })
    
    return pd.DataFrame(records)

hf_df = parse_hf_annotations(hf_annot_path)
print(f"Total annotations: {len(hf_df)}")
hf_df.head()


### 6.3 Label distribution


In [ ]:
hf_df["label"].value_counts()


### 6.4 Visual sanity check (HF images + boxes)


In [ ]:
def plot_hf_sample(n_samples=3):
    sample_imgs = hf_df["image"].dropna().unique()[:n_samples]

    for img_name in sample_imgs:
        # Handle "images/0.png" format - extract just the filename
        img_filename = Path(img_name).name
        img_path = hf_images_dir / img_filename
        if not img_path.exists():
            print("Missing image:", img_path)
            continue

        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h, w, _ = img.shape
        rows = hf_df[hf_df["image"] == img_name]

        for _, row in rows.iterrows():
            x1, y1, x2, y2 = int(row["x_min"]), int(row["y_min"]), int(row["x_max"]), int(row["y_max"])
            label = row["label"]
            color = (255, 0, 0) if "not_free" in label else (0, 255, 0) if "free" in label else (255, 255, 0)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, label, (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"HF sample: {img_filename}")
        plt.show()

plot_hf_sample(3)


## 7. Compiling the Datasets - High-Level Plan

For a simple CNN baseline, we'll create a classification dataset from the HuggingFace dataset by cropping parking space patches.


### 7.1 Cropping HF patches into a classification dataset


In [ ]:
OUT_BASE = BASE_DIR / "data" / "processed" / "cnn_baseline"
for cls in ["free", "not_free", "partial"]:
    (OUT_BASE / cls).mkdir(parents=True, exist_ok=True)

label_map = {
    "free_parking_space": "free",
    "not_free_parking_space": "not_free",
    "partially_free_parking_space": "partial",
}

counter = {"free": 0, "not_free": 0, "partial": 0}

for _, row in tqdm(hf_df.iterrows(), total=len(hf_df)):
    img_name = row["image"]
    label_raw = row["label"]
    cls = label_map.get(label_raw)
    if cls is None:
        continue

    # Handle "images/0.png" format - extract just the filename
    img_filename = Path(img_name).name
    img_path = hf_images_dir / img_filename
    if not img_path.exists():
        continue

    img = cv2.imread(str(img_path))
    if img is None:
        continue

    x1, y1, x2, y2 = map(int, [row["x_min"], row["y_min"], row["x_max"], row["y_max"]])
    patch = img[max(0, y1):y2, max(0, x1):x2]
    if patch.size == 0:
        continue

    counter[cls] += 1
    out_path = OUT_BASE / cls / f"{Path(img_filename).stem}_{counter[cls]:05d}.png"
    cv2.imwrite(str(out_path), patch)

print("\nCropping complete!")
print("Class distribution:")
for cls, count in counter.items():
    print(f"  {cls}: {count} patches")
